# Stock News Research — Incremental Workflow
Run sections in order. Installation is needed only once. Load the same saved run to accumulate data.

This notebook measures text sentiment; it does not yet provide a trained stock forecast. Daily features use a 20:00 America/New_York cutoff and the next observed trading session open-to-close return. Publication and acquisition timestamps are required. The 72-hour freshness window is a research parameter, not an estimated duration of market impact. News coverage is partial; missing sentiment is not neutral sentiment.

Existing schema-version-1 project.json files remain supported. Keep this notebook in the same working folder as your stock_news_data directory, or update DATA_ROOT in section 2. Saved runs remain separate; they are not automatically pooled.


## 1. Install dependencies
Run once; then restart the kernel if required.

In [ ]:
%pip install pandas matplotlib yfinance torch transformers ipywidgets

## 2. Define functions
No downloads or input prompts in this cell. Chinese comments; English interface.

In [ ]:
import re
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timezone
from IPython.display import display, Markdown, clear_output
from matplotlib.ticker import FuncFormatter, PercentFormatter


NEWS_COLUMNS = [
    "Title", "Summary", "Source", "URL", "Published UTC",
    "Fetched UTC", "Added UTC", "Input Type", "Search Match",
    "Relevance Reviewed", "Relevant", "Company", "Target Ticker"
]


def normalize_news(news):

    df = pd.DataFrame() if news is None else news.copy()
    for col in NEWS_COLUMNS:
        if col not in df.columns:
            df[col] = pd.Series(index=df.index, dtype="object")
    for col in ["Title", "Summary", "Source", "URL", "Input Type"]:
        df[col] = df[col].fillna("").astype(str)
    for col in ["Published UTC", "Fetched UTC", "Added UTC"]:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
    df["Published ET"] = df["Published UTC"].dt.tz_convert("America/New_York")
    return df.reset_index(drop=True)


def merge_news(*frames):

    frames = [normalize_news(f) for f in frames if f is not None and not f.empty]
    if not frames:
        return normalize_news(None)
    merged = {}
    for _, row in pd.concat(frames, ignore_index=True).iterrows():
        record = row.to_dict()
        key = ("url", record["URL"].strip()) if record["URL"].strip() else (
            "text", record["Title"].strip().casefold(), record["Summary"].strip().casefold()
        )
        if key not in merged:
            merged[key] = record
            continue
        old = merged[key]
        is_manual = "User-provided" in record["Input Type"]
        old_manual = "User-provided" in old["Input Type"]
        changed = False
        for col in ["Title", "Summary", "Source"]:
            value = record[col].strip()
            if value and value not in {"Untitled user-provided reference", "User-provided", "Unknown"}:
                if is_manual or not old_manual:
                    changed = changed or (col in {"Title", "Summary"} and old[col] != value)
                    old[col] = value
        if changed and not is_manual and pd.notna(record["Fetched UTC"]):
            old["Fetched UTC"] = record["Fetched UTC"]
        for col in ["Published UTC", "Fetched UTC", "Added UTC"]:
            if pd.notna(record[col]) and (pd.isna(old[col]) or is_manual and col != "Fetched UTC"):
                old[col] = record[col]
        types = list(dict.fromkeys((old["Input Type"] + " + " + record["Input Type"]).split(" + ")))
        old["Input Type"] = " + ".join(t for t in types if t)
    return normalize_news(pd.DataFrame(merged.values()))


def parse_articles(articles, route):
    records = []
    fetched = datetime.now(timezone.utc)
    for article in articles:
        content = article.get("content") or article
        if not isinstance(content, dict) or not content.get("title"):
            continue
        provider = content.get("provider") or {}
        source = provider.get("displayName") if isinstance(provider, dict) else str(provider)
        published = pd.to_datetime(content.get("pubDate"), errors="coerce", utc=True)
        if pd.isna(published):
            published = pd.to_datetime(content.get("providerPublishTime"), unit="s", errors="coerce", utc=True)
        def url_value(value):
            return value.get("url", "") if isinstance(value, dict) else ""
        records.append({
            "Title": content["title"], "Summary": content.get("summary") or "",
            "Source": source or content.get("publisher") or "Unknown",
            "URL": url_value(content.get("canonicalUrl")) or url_value(content.get("clickThroughUrl")) or content.get("link") or "",
            "Published UTC": published, "Fetched UTC": fetched,
            "Added UTC": fetched, "Input Type": route
        })
    return merge_news(pd.DataFrame(records))


def fetch_news(config):

    try:
        articles = yf.Search(config["company"], news_count=20, max_results=5).news or []
        news = parse_articles(articles, "Company search")
    except Exception:
        print("Company search failed. Trying ticker news.")
        news = normalize_news(None)
    if news.empty:
        try:
            articles = yf.Ticker(config["ticker"]).get_news(count=30, tab="news") or []
            news = parse_articles(articles, "Ticker news")
        except Exception:
            print("Ticker news retrieval failed. Manual sources are still available.")
    news["Company"] = config["company"]
    news["Target Ticker"] = config["ticker"]
    print(f"News candidates retrieved: {len(news)}")
    print("Search results require review; historical coverage is not guaranteed.")
    return news


def fetch_market(config):
    try:
        df = yf.download(config["ticker"], period=config["period"], interval="1d",
                         auto_adjust=True, progress=False)
    except Exception:
        print("Market download failed. You can still continue with news.")
        return pd.DataFrame()
    if df.empty:
        print("No market data returned. Verify the ticker and company identity.")
        return df
    if isinstance(df.columns, pd.MultiIndex):
        level = next((i for i in range(df.columns.nlevels)
                      if config["ticker"] in df.columns.get_level_values(i)), None)
        if level is None:
            print("Unexpected market data format.")
            return pd.DataFrame()
        df = df.xs(config["ticker"], axis=1, level=level).copy()
    fields = ["Open", "High", "Low", "Close", "Volume"]
    if not set(fields).issubset(df.columns):
        print("Required market fields are missing.")
        return pd.DataFrame()
    df = df.sort_index()
    df = df.loc[~df.index.duplicated(keep="last")].copy()
    for col in fields:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=fields)
    df = df.loc[(df["Open"] > 0) & (df["Close"] > 0)].copy()
    df["MA5"] = df["Close"].rolling(5).mean()
    df["MA20"] = df["Close"].rolling(20).mean()
    df["VolumeMA20"] = df["Volume"].rolling(20).mean()
    df["DailyReturn"] = df["Close"].pct_change(fill_method=None)
    df["IntradayReturn"] = df["Close"] / df["Open"] - 1
    return df


def plot_market(df, config):

    if df is None or len(df) < 2:
        print("At least two market records are needed for the dashboard.")
        return
    latest = float(df["Close"].iloc[-1])
    daily = float(df["DailyReturn"].iloc[-1])
    change = latest / float(df["Close"].iloc[0]) - 1
    colors = ["#94a3b8" if pd.isna(r) else "#16a34a" if r >= 0 else "#dc2626" for r in df["DailyReturn"]]
    fig, axes = plt.subplots(3, 1, figsize=(11, 5), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1, 1]}, constrained_layout=True)
    fig.suptitle(f"{config['company']} ({config['ticker']}) | Market Dashboard\n"
                 f"Latest adj. close: {latest:,.2f} | Daily: {daily:+.2%} | Period: {change:+.2%}", fontsize=11)
    for col, color, label in [("Close", "#2563eb", "Adjusted Close"),
                               ("MA5", "#f59e0b", "5-Day MA"), ("MA20", "#9333ea", "20-Day MA")]:
        axes[0].plot(df.index, df[col], color=color, label=label, linewidth=1.3)
    axes[0].set_ylabel("Adjusted Price")
    axes[0].legend(ncol=3, fontsize=8, loc="upper left")
    axes[1].bar(df.index, df["Volume"], color=colors, width=0.8, alpha=0.7)
    axes[1].plot(df.index, df["VolumeMA20"], color="#334155", linewidth=1)
    axes[1].set_ylabel("Volume")
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1e6:.1f}M"))
    axes[2].bar(df.index, df["DailyReturn"], color=colors, width=0.8)
    axes[2].axhline(0, color="#64748b", linewidth=0.7)
    axes[2].set_ylabel("Return")
    axes[2].yaxis.set_major_formatter(PercentFormatter(1))
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    axes[2].xaxis.set_major_locator(locator)
    axes[2].xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    for ax in axes:
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
        ax.tick_params(labelsize=8)
    plt.show()
    print(f"{len(df)} records | {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}")
    print("Prices are adjusted; the latest session may be incomplete.")

def ask_yes_no(prompt):

    while True:
        answer = input(prompt).strip().lower()

        if answer in {"y", "yes"}:
            return True

        if answer in {"n", "no"}:
            return False

        print("Please enter y or n.")

def read_multiline_text():

    print("Paste the article excerpt or information below.")
    print("Type END on a separate line to finish.")
    print("For a link-only reference, type END immediately.")

    lines = []

    while True:
        line = input()

        if line.strip() == "END":
            break

        lines.append(line)

    return "\n".join(lines).strip()

def collect_additional_sources():
    records = []


    add_more = ask_yes_no(
        "Would you like to add news, links, or other information? [y/n]: "
    )

    while add_more:
        print(f"\nAdditional Source {len(records) + 1}")


        while True:
            url = input(
                "Source URL (press Enter if unavailable): "
            ).strip()

            if not url or url.startswith(("https://", "http://")):
                break

            print("Please enter a URL starting with https:// or http://.")

        source = input(
            "Source / publisher name [User-provided]: "
        ).strip() or "User-provided"

        title = input(
            "Article title or information label: "
        ).strip()

        while True:
            published_input = input(
                "Publication time with timezone "
                "(e.g., 2026-10-08T14:30:00-04:00; "
                "press Enter if unknown): "
            ).strip()

            if not published_input:
                published = pd.NaT
                break

            try:
                timestamp = pd.Timestamp(published_input)

                if pd.isna(timestamp) or timestamp.tzinfo is None:
                    print("Please include a timezone, or leave this blank.")
                    continue

                published = timestamp.tz_convert("UTC")
                break

            except (ValueError, TypeError):
                print("Invalid timestamp. Please try again.")

        text = read_multiline_text()

        if not url and not title and not text:
            print("Empty entry skipped.")
        else:
            records.append({
                "Title": title or "Untitled user-provided reference",
                "Summary": text,
                "Source": source,
                "URL": url,
                "Published UTC": published,
                "Fetched UTC": pd.NaT,
                "Added UTC": datetime.now(timezone.utc),
                "Input Type": "User-provided",
                "Search Match": False,
                "Relevance Reviewed": False
            })

            print("Source added.")

        add_more = ask_yes_no(
            "Would you like to add another source? [y/n]: "
        )

    return pd.DataFrame(records)





from pathlib import Path
from io import StringIO
import hashlib
import json
import os
import uuid

DATA_ROOT = (Path.cwd() / "stock_news_data").resolve()
DATASETS = ["market", "news_raw", "manual_sources", "news_reviewed", "review_log"]


def ticker_folder(ticker):
 
    ticker = ticker.strip().upper()
    slug = re.sub(r"[^A-Z0-9._-]", "_", ticker).strip(".") or "TICKER"
    digest = hashlib.sha256(ticker.encode()).hexdigest()[:8]
    return DATA_ROOT / f"{slug}_{digest}"


def new_run(config):

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
    return {
        "config": config.copy(), "run_id": run_id,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "market": pd.DataFrame(), "news_raw": normalize_news(None),
        "manual_sources": normalize_news(None), "news_reviewed": normalize_news(None),
        "review_log": normalize_news(None), "review_complete": False,
        "news_fetch_attempted": False, "market_fetch_attempted": False
    }


def run_path(project):
   
    run_id = project["run_id"]
    if not re.fullmatch(r"\d{8}T\d{6}Z_[a-f0-9]{8}", run_id):
        raise ValueError("Invalid run ID.")
    return ticker_folder(project["config"]["ticker"]) / run_id / "project.json"


def save_project(project):

    path = run_path(project)
    path.parent.mkdir(parents=True, exist_ok=True)
    config = project["config"]
    data = {}
    for name in DATASETS:
        frame = project[name].copy()
        frame["Company"] = config["company"]
        frame["Target Ticker"] = config["ticker"]
        frame["Run ID"] = project["run_id"]
        data[name] = json.loads(frame.to_json(orient="split", date_format="iso", date_unit="ms"))
    payload = {
        "schema_version": 1, "config": config, "run_id": project["run_id"],
        "created_utc": project["created_utc"],
        "saved_utc": datetime.now(timezone.utc).isoformat(),
        "review_complete": bool(project["review_complete"]),
        "news_fetch_attempted": bool(project.get("news_fetch_attempted", False)),
        "market_fetch_attempted": bool(project.get("market_fetch_attempted", False)),
        "datasets": data
    }
    temporary = path.with_name(".project_" + uuid.uuid4().hex + ".tmp")
    try:
        with temporary.open("w", encoding="utf-8") as handle:
            json.dump(payload, handle, ensure_ascii=False, allow_nan=False, indent=2)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)
    print(f"Saved run: {project['run_id']}")
    print(f"Location: {path.parent}")
    return path


def load_project(path, expected_ticker):
    payload = json.loads(Path(path).read_text(encoding="utf-8"))
    if payload.get("schema_version") != 1:
        raise ValueError("Unsupported saved project format.")
    if payload["config"]["ticker"] != expected_ticker.strip().upper():
        raise ValueError("Saved ticker does not match the requested ticker.")
    result = {key: payload[key] for key in ["config", "run_id", "created_utc", "review_complete"]}
    for flag in ["news_fetch_attempted", "market_fetch_attempted"]:
        result[flag] = bool(payload.get(flag, False))
    for name in DATASETS:
        frame = pd.read_json(StringIO(json.dumps(payload["datasets"][name])), orient="split", convert_dates=False)
        for column, expected in [("Target Ticker", result["config"]["ticker"]),
                                 ("Company", result["config"]["company"]), ("Run ID", result["run_id"])]:
            if column not in frame or not frame[column].eq(expected).all():
                raise ValueError("Saved dataset identity is inconsistent.")
        if name == "market":
            if not frame.empty:
                frame.index = pd.to_datetime(frame.index, errors="raise")
                frame.index.name = "Date"
        else:
            frame = normalize_news(frame)
        result[name] = frame
    if run_path(result).resolve() != Path(path).resolve():
        raise ValueError("Saved run identity does not match its folder.")
    return result


def choose_project():
    ticker = input("Ticker symbol: ").strip().upper()
    while not ticker:
        ticker = input("Please enter a ticker symbol: ").strip().upper()
    saved = sorted(ticker_folder(ticker).glob("*/project.json"), reverse=True)
    if saved:
        print("Saved analyses (up to 10 most recent):")
        choices = saved[:10]
        for i, path in enumerate(choices, 1):
            try:
                header = json.loads(path.read_text(encoding="utf-8"))
                print(f"{i}. {header['config']['company']} | {header['run_id']} | Review complete: {header['review_complete']}")
            except (OSError, ValueError, KeyError):
                print(f"{i}. Unreadable saved analysis")
        while True:
            choice = input("Enter a number to load, or N for a new analysis: ").strip().lower()
            if choice == "n":
                break
            if choice.isdigit() and 1 <= int(choice) <= len(choices):
                try:
                    result = load_project(choices[int(choice)-1], ticker)
                    print(f"Loaded: {result['config']['company']} ({ticker}) | {result['run_id']}")
                    return result
                except (OSError, ValueError, KeyError, TypeError):
                    print("Could not load this analysis. Choose another record or start new.")
            else:
                print("Please select a listed number or N.")
    company = input("Company name: ").strip()
    while not company:
        company = input("Please enter a company name: ").strip()
    period = input("Period: 1mo / 3mo / 6mo / 1y / 5y / max [6mo]: ").strip().lower() or "6mo"
    if period not in {"1mo", "3mo", "6mo", "1y", "5y", "max"}:
        print("Invalid period. Using 6mo.")
        period = "6mo"
    default = "Starlink, Starship" if company.casefold() == "spacex" else ""
    entry = input(f"Additional keywords, comma-separated [{default}]: ").strip() or default
    result = new_run({"company": company, "ticker": ticker, "period": period,
                      "keywords": [k.strip() for k in entry.split(",") if k.strip()]})
    save_project(result)
    print("New analysis started. Verify that the ticker represents the intended company.")
    return result

def analysis_text(row):
    title = str(row.get("Title", "") or "").strip()
    if title == "Untitled user-provided reference":
        title = ""
    summary = row.get("Summary", "")
    summary = "" if pd.isna(summary) else str(summary).strip()
    return ". ".join(x for x in [title, summary] if x)


def article_key(row):
    url = str(row.get("URL", "") or "").strip()
    return "url:" + url if url else "title:" + str(row.get("Title", "")).strip().casefold()


def relevance_text(row):

    return json.dumps([str(row.get(c, "")) for c in ["Title", "Summary", "Source", "URL"]], ensure_ascii=False)


def sync_aliases(project):
    globals().update(config=project["config"], stock_data=project["market"],
                     news_data=project["news_raw"], reviewed_news=project["news_reviewed"],
                     review_log=project["review_log"])
    for name in ["daily_features", "current_news", "training_candidates", "sentiment_evaluation"]:
        globals().pop(name, None)


def review_project_sources(project):

    combined = merge_news(project["news_raw"], project["manual_sources"])
    history = normalize_news(project["review_log"])
    retained = normalize_news(project["news_reviewed"])
    prior = {}
    for _, row in pd.concat([retained, history], ignore_index=True).iterrows():
        status = row.get("Decision", "")
        if status not in {"Accepted", "Rejected"}:
            status = row.get("Review Status", "")
        if status not in {"Accepted", "Rejected"}:
            if row.get("Relevant") == True:
                status = "Accepted"
            else:
                continue
        record = row.to_dict()
        record["Review Status"] = status
        prior[article_key(row)] = record
    accepted = {article_key(row): row.to_dict() for _, row in retained.iterrows()}
    project["review_complete"] = False
    save_project(project)
    config = project["config"]
    for position, (_, candidate) in enumerate(combined.iterrows(), 1):
        key = article_key(candidate)
        old = prior.get(key)
        fingerprint = relevance_text(candidate)
        old_fingerprint = old.get("Relevance Text", relevance_text(old)) if old else None
        if old and fingerprint == old_fingerprint:
            keep = old["Review Status"] == "Accepted"
        else:
            clear_output(wait=True)
            print(f"Reference {position} of {len(combined)} | {config['company']} ({config['ticker']})")
            print("TITLE\n" + candidate["Title"])
            print("SUMMARY\n" + (candidate["Summary"] or "No excerpt available."))
            print("Source: " + candidate["Source"])
            print("URL: " + candidate["URL"])
            print("A matching name alone does not establish company relevance.")
            answer = input("Keep this reference? Y / N / Q: ").strip().upper()
            while answer not in {"Y", "N", "Q"}:
                answer = input("Please enter Y, N, or Q: ").strip().upper()
            if answer == "Q":
                sync_aliases(project)
                print("Review paused. Completed decisions are saved. Run this section to resume.")
                return
            keep = answer == "Y"
        record = candidate.to_dict()
        if key in accepted:
            preserved = accepted[key].copy()
            preserved.update(record)
            record = preserved
        record.update({"Relevant": keep, "Relevance Reviewed": True,
                       "Review Status": "Accepted" if keep else "Rejected",
                       "Decision": "Accepted" if keep else "Rejected",
                       "Relevance Text": fingerprint})
        if old:
            for column in ["Exclusion Reason", "Excluded UTC"]:
                if column in old:
                    record[column] = old[column]
        if keep:
            accepted[key] = record
        else:
            accepted.pop(key, None)
        prior[key] = record
        project["news_reviewed"] = normalize_news(pd.DataFrame(accepted.values()))
        project["review_log"] = normalize_news(pd.DataFrame(prior.values()))
        save_project(project)
    project["review_complete"] = True
    sync_aliases(project)
    save_project(project)
    print(f"Retained references: {len(project['news_reviewed'])}")


def score_reviewed_news(project):
    if not project.get("review_complete"):
        print("Complete relevance review first.")
        return
    news = normalize_news(project["news_reviewed"])
    if news.empty:
        print("No retained news available.")
        return
    texts = news.apply(analysis_text, axis=1)
    for c in ["Analysis Text", "Sentiment", "Sentiment Model"]:
        if c not in news:
            news[c] = ""
    for c in ["Positive", "Negative", "Neutral", "Sentiment Score"]:
        if c not in news:
            news[c] = float("nan")
        news[c] = pd.to_numeric(news[c], errors="coerce")
    reusable = texts.eq(news["Analysis Text"].fillna("")) & news["Sentiment Model"].eq("ProsusAI/finbert")
    reusable &= news["Sentiment"].isin(["Positive", "Negative", "Neutral"])
    reusable &= news["Sentiment Score"].between(-1, 1)
    for c in ["Positive", "Negative", "Neutral"]:
        reusable &= news[c].between(0, 1)
    reusable &= (news[["Positive", "Negative", "Neutral"]].sum(axis=1)-1).abs().lt(1e-4)
    pending = news.index[texts.ne("") & ~reusable].tolist()
    print(f"Scores reused: {int((reusable & texts.ne('')).sum())} | Articles to score: {len(pending)}")
    if pending:
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        global finbert_tokenizer, finbert_model
        if "finbert_model" not in globals() or "finbert_tokenizer" not in globals():
            print("Loading FinBERT. First use requires a model download.")
            finbert_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
            finbert_model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
        finbert_model.eval()
        labels = [str(finbert_model.config.id2label[i]).lower() for i in range(finbert_model.config.num_labels)]
        if set(labels) != {"positive", "negative", "neutral"}:
            raise ValueError("Unexpected sentiment labels.")
        for start in range(0, len(pending), 4):
            indices = pending[start:start+4]
            inputs = finbert_tokenizer(texts.loc[indices].tolist(), padding=True, truncation=True,
                                       max_length=512, return_tensors="pt")
            with torch.inference_mode():
                values = torch.softmax(finbert_model(**inputs).logits, dim=-1).cpu().numpy()
            for idx, probabilities in zip(indices, values):
                scores = dict(zip(labels, map(float, probabilities)))
                for label, value in scores.items():
                    news.at[idx, label.capitalize()] = value
                news.at[idx, "Sentiment"] = max(scores, key=scores.get).capitalize()
                news.at[idx, "Sentiment Score"] = scores["positive"]-scores["negative"]
                news.at[idx, "Analysis Text"] = texts.at[idx]
                news.at[idx, "Sentiment Model"] = "ProsusAI/finbert"
    blank = texts.eq("")
    news.loc[blank, ["Positive", "Negative", "Neutral", "Sentiment Score"]] = float("nan")
    news.loc[blank, "Sentiment"] = "Not scored"
    news.loc[blank, "Analysis Text"] = ""
    project["news_reviewed"] = news
    sync_aliases(project)
    save_project(project)
    print("Text sentiment only. Human labels do not overwrite model scores.")


def build_daily_features(project, freshness_hours=72):
    if not project:
        print("Start or load a project first.")
        return pd.DataFrame()

    if freshness_hours <= 0:
        raise ValueError("Freshness hours must be positive.")

    if not project.get("review_complete"):
        print("Complete the news review first.")
        return pd.DataFrame()

    market = project["market"].copy()
    news = project["news_reviewed"].copy()

    if market.empty or not {"Open", "Close"}.issubset(market.columns):
        print("Load market data first.")
        return pd.DataFrame()

    if news.empty or "Sentiment Score" not in news.columns:
        print("Complete sentiment analysis first.")
        return pd.DataFrame()

    market = market.sort_index()
    dates = pd.DatetimeIndex(market.index)

    if dates.tz is not None:
        dates = dates.tz_convert(
            "America/New_York"
        ).tz_localize(None)

    dates = dates.normalize()

    if dates.duplicated().any():
        raise ValueError("Duplicate trading dates found.")

    market.index = dates
    market.index.name = "Trading Date"

    cutoffs = (
        dates + pd.Timedelta(hours=20)
    ).tz_localize("America/New_York")

    now = pd.Timestamp.now(tz="America/New_York")
    completed = cutoffs <= now

    market = market.loc[completed].copy()
    cutoffs = cutoffs[completed]

    if len(market) < 2:
        print("At least two completed prediction dates are required.")
        return pd.DataFrame()

    def time_column(name):
        if name not in news.columns:
            return pd.Series(
                pd.NaT,
                index=news.index,
                dtype="datetime64[ns, UTC]"
            )

        return pd.to_datetime(
            news[name],
            errors="coerce",
            utc=True
        )

    published = time_column("Published UTC")
    fetched = time_column("Fetched UTC")
    added = time_column("Added UTC")


    input_type = news.get(
        "Input Type",
        pd.Series("", index=news.index)
    ).fillna("").astype(str)

    manual = input_type.str.contains(
        "User-provided",
        regex=False
    )

    acquired = fetched.fillna(added)
    acquired.loc[manual] = added.loc[manual]

    scores = pd.to_numeric(
        news["Sentiment Score"],
        errors="coerce"
    )

    timing_valid = (
        published.notna()
        & acquired.notna()
        & (published <= acquired)
        & (acquired <= now.tz_convert("UTC"))
    )

    freshness = pd.Timedelta(hours=freshness_hours)

    news["Prediction Use"] = "Timing needs verification"

    too_old_at_acquisition = (
        timing_valid
        & ((acquired - published) > freshness)
    )

    news.loc[
        too_old_at_acquisition,
        "Prediction Use"
    ] = "Historical Reference"

    eligible = (
        timing_valid
        & ~too_old_at_acquisition
        & scores.notna()
    )

    news.loc[
        timing_valid & ~too_old_at_acquisition & scores.isna(),
        "Prediction Use"
    ] = "Sentiment not scored"

    news.loc[
        eligible,
        "Prediction Use"
    ] = "Eligible at acquisition"

    news["Acquired UTC"] = acquired
    news["Sentiment Score"] = scores


    daily = market.copy()
    daily["Prediction Cutoff ET"] = cutoffs

    for column in [
        "News Count",
        "Mean News Sentiment",
        "Negative News Share",
        "Has News"
    ]:
        daily[column] = float("nan")

    used_indices = set()


    for i in range(1, len(daily)):
        previous = cutoffs[i - 1].tz_convert("UTC")
        current = cutoffs[i].tz_convert("UTC")

        in_window = (
            eligible
            & (acquired > previous)
            & (acquired <= current)
            & (published >= current - freshness)
            & (published <= current)
        )

        window = news.loc[in_window]
        used_indices.update(window.index.tolist())

        date = daily.index[i]
        count = len(window)

        daily.loc[date, "News Count"] = count
        daily.loc[date, "Has News"] = int(count > 0)

        if count:
            daily.loc[date, "Mean News Sentiment"] = (
                window["Sentiment Score"].mean()
            )


            if {"Positive", "Negative", "Neutral"}.issubset(
                window.columns
            ):
                probabilities = window[
                    ["Positive", "Negative", "Neutral"]
                ].apply(pd.to_numeric, errors="coerce")

                valid_labels = probabilities.notna().all(axis=1)

                if valid_labels.any():
                    daily.loc[date, "Negative News Share"] = (
                        probabilities.loc[valid_labels]
                        .idxmax(axis=1)
                        .eq("Negative")
                        .mean()
                    )

    daily["News Coverage"] = "Unknown / partial collected feed"


    intraday = daily["Close"] / daily["Open"] - 1
    daily["Next Session Return"] = intraday.shift(-1)

    daily["Target Up"] = pd.Series(
        pd.NA,
        index=daily.index,
        dtype="Int64"
    )

    known_target = daily["Next Session Return"].notna()

    daily.loc[known_target, "Target Up"] = (
        daily.loc[known_target, "Next Session Return"] > 0
    ).astype(int)


    project["news_reviewed"] = news
    save_project(project)

    print(f"Freshness window: {freshness_hours} hours")
    print(f"Completed prediction dates: {len(daily)}")
    print(
        "Historical references:",
        int(news["Prediction Use"].eq("Historical Reference").sum())
    )
    print(f"Articles used in displayed historical windows: {len(used_indices)}")
    print(
        "Dates containing collected news:",
        int(daily["Has News"].eq(1).sum())
    )

    print(
        "Zero collected articles does not prove no market news existed. "
        "This dataset is not ready for prediction training yet."
    )

    display(
        daily[[
            "Close",
            "News Count",
            "Mean News Sentiment",
            "Has News",
            "Next Session Return",
            "Target Up"
        ]].tail(5).style.format({
            "Close": "{:.2f}",
            "News Count": "{:.0f}",
            "Mean News Sentiment": "{:+.3f}",
            "Has News": "{:.0f}",
            "Next Session Return": "{:+.2%}",
            "Target Up": "{:.0f}"
        }, na_rep="—")
    )

    return daily

def show_current_news_summary(project, freshness_hours=72):

    if not project:
        print("Start or load a project first.")
        return pd.DataFrame()

    if not project.get("review_complete"):
        print("Complete the news review first.")
        return pd.DataFrame()

    if freshness_hours <= 0:
        raise ValueError("Freshness hours must be positive.")

    news = project["news_reviewed"].copy()

    required = {"Sentiment Score", "Sentiment", "Title"}

    if news.empty or not required.issubset(news.columns):
        print("Complete sentiment analysis first.")
        return pd.DataFrame()


    now_utc = pd.Timestamp.now(tz="UTC")
    now_et = now_utc.tz_convert("America/New_York")
    window_start = now_utc - pd.Timedelta(hours=freshness_hours)

    def time_column(name):
        if name not in news.columns:
            return pd.Series(
                pd.NaT,
                index=news.index,
                dtype="datetime64[ns, UTC]"
            )

        return pd.to_datetime(
            news[name],
            errors="coerce",
            utc=True
        )

    published = time_column("Published UTC")
    fetched = time_column("Fetched UTC")
    added = time_column("Added UTC")


    input_type = news.get(
        "Input Type",
        pd.Series("", index=news.index)
    ).fillna("").astype(str)

    manual = input_type.str.contains(
        "User-provided",
        regex=False
    )

    acquired = fetched.fillna(added)
    acquired.loc[manual] = added.loc[manual]

    scores = pd.to_numeric(
        news["Sentiment Score"],
        errors="coerce"
    )


    eligible = (
        published.notna()
        & acquired.notna()
        & (published <= acquired)
        & (acquired <= now_utc)
        & (published >= window_start)
        & (published <= now_utc)
        & scores.notna()
        & scores.between(-1, 1)
        & news["Sentiment"].isin(["Positive", "Neutral", "Negative"])
    )

    current = news.loc[eligible].copy()

    current["Sentiment Score"] = scores.loc[eligible]
    current["Published ET"] = (
        published.loc[eligible]
        .dt.tz_convert("America/New_York")
    )
    current["Age Hours"] = (
        (now_utc - published.loc[eligible])
        .dt.total_seconds() / 3600
    )

    current = current.sort_values(
        "Published ET",
        ascending=False
    )

    config = project["config"]

    print(f"Company: {config['company']} ({config['ticker']})")
    print(f"As of: {now_et:%Y-%m-%d %H:%M:%S %Z}")
    print(f"Freshness window: {freshness_hours} hours")
    print(f"Eligible articles: {len(current)} of {len(news)} retained")

    if current.empty:
        print("No retained, scored articles qualify at this time.")
        print("This does not mean no relevant market news exists.")
        return current


    counts = (
        current["Sentiment"]
        .value_counts()
        .reindex(
            ["Positive", "Neutral", "Negative"],
            fill_value=0
        )
    )

    mean_score = current["Sentiment Score"].mean()


    fig, axes = plt.subplots(
        1, 2,
        figsize=(10, 3.2),
        gridspec_kw={"width_ratios": [1.1, 1]},
        constrained_layout=True
    )

    ax_summary, ax_counts = axes


    ax_summary.axis("off")

    ax_summary.text(
        0.03, 0.96,
        "Current News Snapshot",
        transform=ax_summary.transAxes,
        fontsize=14,
        fontweight="bold",
        ha="left",
        va="top"
    )

    summary_lines = [
        f"Eligible articles: {len(current)}",
        f"Mean text sentiment: {mean_score:+.3f}",
        f"Window: Last {freshness_hours} hours"
    ]

    for y, text in zip([0.72, 0.54, 0.36], summary_lines):
        ax_summary.text(
            0.03, y,
            text,
            transform=ax_summary.transAxes,
            fontsize=11,
            ha="left",
            va="top"
        )

    ax_summary.text(
        0.03, 0.13,
        "Text sentiment only\nNot a stock price prediction",
        transform=ax_summary.transAxes,
        fontsize=9,
        color="#64748b",
        ha="left",
        va="top"
    )


    ax_counts.bar(
        counts.index,
        counts.values,
        color=["#16a34a", "#94a3b8", "#dc2626"],
        width=0.65
    )

    ax_counts.set_title(
        "Sentiment Classification",
        fontsize=12
    )
    ax_counts.set_ylabel("Articles")
    ax_counts.set_ylim(0, max(int(counts.max()) + 1, 1))

    ax_counts.set_yticks(
        range(0, int(counts.max()) + 2)
    )

    ax_counts.set_axisbelow(True)
    ax_counts.grid(axis="y", alpha=0.2)
    ax_counts.spines["top"].set_visible(False)
    ax_counts.spines["right"].set_visible(False)

    for i, count in enumerate(counts):
        ax_counts.text(
            i, count + 0.08,
            str(count),
            ha="center",
            va="bottom",
            fontsize=10
        )

    plt.show()


    preview = current[[
        "Published ET",
        "Title",
        "Sentiment",
        "Sentiment Score",
        "Age Hours"
    ]].head(5).copy()

    preview["Published ET"] = preview["Published ET"].dt.strftime(
        "%m-%d %H:%M %Z"
    )


    preview["Title"] = preview["Title"].apply(
        lambda title: (
            str(title)[:107] + "..."
            if len(str(title)) > 110
            else str(title)
        )
    )

    display(
        preview.style.format({
            "Sentiment Score": "{:+.3f}",
            "Age Hours": "{:.1f}"
        })
        .hide(axis="index")
        .set_properties(**{
            "font-size": "12px",
            "text-align": "left"
        })
    )

    print(
        "Showing up to 5 articles. "
        "All eligible articles and full titles are stored in current_news."
    )
    print(
        "Model labels are unchanged. "
        "Review questionable classifications using the full analysis text."
    )
    print(
        "Coverage is partial. Repeated snapshots are not "
        "independent training samples."
    )

    return current

def review_news_sentiment(project):

    if not isinstance(project, dict):
        print("Please load your project first.")
        return

    news = project.get("news_reviewed")

    if not isinstance(news, pd.DataFrame) or news.empty:
        print("No reviewed news available.")
        return

    if "Title" not in news.columns:
        print("The Title column is missing.")
        return

    save_function = globals().get("save_project")

    if not callable(save_function):
        print("Please run the cell that defines save_project first.")
        return


    news = news.copy().reset_index(drop=True)

    review_columns = [
        "Human Sentiment",
        "Review Reason",
        "Human Reviewed UTC",
        "Human Reviewed Text"
    ]

    for column in review_columns:
        if column not in news.columns:
            news[column] = ""
        news[column] = news[column].fillna("").astype("object")

    labels = {
        "P": "Positive",
        "N": "Negative",
        "U": "Neutral"
    }


    def get_review_text(row):
        analysis_text = row.get("Analysis Text", "")
        if pd.notna(analysis_text) and str(analysis_text).strip():
            return str(analysis_text).strip()

        parts = []
        for column in ["Title", "Summary"]:
            value = row.get(column, "")
            if pd.notna(value) and str(value).strip():
                parts.append(str(value).strip())
        return "\n\n".join(parts)

    texts = news.apply(get_review_text, axis=1)


    reviewed = news["Human Sentiment"].isin(labels.values())
    same_text = news["Human Reviewed Text"].eq(texts)
    pending_indices = news.index[~(reviewed & same_text)].tolist()

    if not pending_indices:
        print("All current articles have been reviewed.")
        return

    print(f"Articles awaiting review: {len(pending_indices)}")
    print("Judge the financial tone of the available text.")
    print("P = Positive | N = Negative | U = Neutral")
    print("S = Skip | Q = Quit")

    input("Press Enter to start: ")

    saved_count = 0

    for position, index in enumerate(pending_indices, start=1):
        clear_output(wait=True)
        row = news.loc[index]

        print(f"ARTICLE {position} OF {len(pending_indices)}")
        print(f"Title: {row['Title']}")

        for column in ["Source", "URL"]:
            value = row.get(column, "")
            if pd.notna(value) and str(value).strip():
                print(f"{column}: {value}")

        print("\nREVIEW TEXT")
        print(texts.loc[index])
        print("\nP = Positive | N = Negative | U = Neutral")
        print("S = Skip | Q = Quit")


        while True:
            choice = input("\nYour label: ").strip().upper()
            if choice in {*labels, "S", "Q"}:
                break
            print("Please enter P, N, U, S, or Q.")

        if choice == "Q":
            print("Review stopped. Previously saved reviews are retained.")
            break

        if choice == "S":
            continue

        reason = input("Reason (optional): ").strip()

        news.at[index, "Human Sentiment"] = labels[choice]
        news.at[index, "Review Reason"] = reason
        news.at[index, "Human Reviewed UTC"] = (
            pd.Timestamp.now(tz="UTC").isoformat()
        )
        news.at[index, "Human Reviewed Text"] = texts.loc[index]

        project["news_reviewed"] = news
        globals()["reviewed_news"] = news

        try:
            save_function(project)
        except Exception as error:
            print(f"Save failed: {error}")
            print("The latest review remains in memory.")
            print("Run save_project(project) before closing the notebook.")
            return

        saved_count += 1
        print(f"\nHuman label: {labels[choice]}")

        model_label = row.get("Sentiment", "")
        if pd.notna(model_label) and str(model_label).strip():
            print(f"Model label: {model_label}")

        input("Review saved. Press Enter to continue: ")

    print(f"\nReviews saved in this session: {saved_count}")


    columns = [
        "Title", "Sentiment", "Human Sentiment", "Review Reason"
    ]
    columns = [column for column in columns if column in news.columns]

    audited = news["Human Sentiment"].isin(labels.values())
    display(news.loc[audited, columns])

def evaluate_sentiment_reviews(project):

    if not isinstance(project, dict):
        print("Please load your project first.")
        return

    news = project.get("news_reviewed")

    if not isinstance(news, pd.DataFrame) or news.empty:
        print("No news available.")
        return

    required = ["Sentiment", "Human Sentiment"]
    missing = [column for column in required if column not in news.columns]

    if missing:
        print("Missing columns:", ", ".join(missing))
        print("Please complete the model scoring and human review first.")
        return


    labels = ["Positive", "Neutral", "Negative"]
    model_labels = news["Sentiment"].fillna("").astype(str).str.strip().str.title()
    human_labels = news["Human Sentiment"].fillna("").astype(str).str.strip().str.title()

    valid = model_labels.isin(labels) & human_labels.isin(labels)


    if {"Analysis Text", "Human Reviewed Text"}.issubset(news.columns):
        current_text = news["Analysis Text"].fillna("").astype(str).str.strip()
        reviewed_text = (
            news["Human Reviewed Text"].fillna("").astype(str).str.strip()
        )
        valid &= current_text.ne("") & current_text.eq(reviewed_text)

    compared = news.loc[valid].copy()
    compared["Model Label"] = model_labels.loc[valid]
    compared["Human Label"] = human_labels.loc[valid]

    if compared.empty:
        print("No comparable reviews available.")
        print("Review articles with valid model labels and unchanged input text.")
        return

    agreement = compared["Model Label"].eq(compared["Human Label"])
    total = len(compared)
    matched = int(agreement.sum())

    print("SENTIMENT REVIEW RESULTS")
    print(f"Articles compared: {total}")
    print(f"Matching labels: {matched}")
    print(f"Different labels: {total - matched}")
    print(f"Label agreement: {matched / total:.1%}")
    print("This measures sentiment label agreement, not stock prediction accuracy.")


    matrix = pd.crosstab(
        compared["Human Label"],
        compared["Model Label"]
    ).reindex(index=labels, columns=labels, fill_value=0)

    matrix.index.name = "Human Label"
    matrix.columns.name = "Model Label"

    print("\nLABEL COMPARISON TABLE")
    display(matrix)


    disagreements = compared.loc[~agreement]
    columns = [
        "Title", "Model Label", "Human Label",
        "Sentiment Score", "Review Reason"
    ]
    columns = [
        column for column in columns if column in compared.columns
    ]

    print("\nARTICLES WITH DIFFERENT LABELS")
    if disagreements.empty:
        print("No disagreements in the reviewed sample.")
    else:
        display(disagreements[columns])

    print("\nThis small reviewed sample is exploratory.")
    print("Human labels may also contain errors.")
    print("The previously inspected article was selected after seeing its model label.")

    return compared

def check_training_readiness(daily_features):

    if not isinstance(daily_features, pd.DataFrame):
        print("Please run the daily feature generation cell first.")
        return

    if daily_features.empty:
        print("No daily features available.")
        return

    data = daily_features.copy()


    print("AVAILABLE DAILY FEATURE COLUMNS")
    print(", ".join(map(str, data.columns)))


    target_candidates = [
        "Next Session Return",
        "Next Day Return",
        "Next Trading Day Return"
    ]

    target_column = next(
        (column for column in target_candidates if column in data.columns),
        None
    )

    required_features = ["News Count", "Mean News Sentiment"]
    missing = [
        column for column in required_features
        if column not in data.columns
    ]

    if missing or target_column is None:
        print("\nTraining readiness could not be checked.")
        if missing:
            print("Missing feature columns:", ", ".join(missing))
        if target_column is None:
            print("No recognized next-session return column found.")
        return


    news_count = pd.to_numeric(data["News Count"], errors="coerce")
    sentiment = pd.to_numeric(data["Mean News Sentiment"], errors="coerce")
    target = pd.to_numeric(data[target_column], errors="coerce")

    has_news = news_count.gt(0) & sentiment.notna()
    observed_target = target.notna()
    eligible = has_news & observed_target

    print("\nTRAINING DATA STATUS")
    print(f"Trading days in dataset: {len(data)}")
    print(f"Days with news sentiment: {int(has_news.sum())}")
    print(f"Days with observed next-session returns: {int(observed_target.sum())}")
    print(f"Candidate training days: {int(eligible.sum())}")

    columns = [
        "News Count",
        "Mean News Sentiment",
        "Negative News Share",
        target_column,
        "Target Up"
    ]
    columns = [column for column in columns if column in data.columns]

    print("\nRECENT DAILY DATA")
    display(data[columns].tail(10))

    if not eligible.any():
        print("\nNo candidate training days yet.")
        print("Keep collecting news and refresh prices after later sessions.")
        print("Do not move newly acquired news into past trading days.")
    else:
        print("\nCandidate rows are available for further validation.")
        print("Check collection coverage before training.")
        print("Use chronological training and test periods.")


    return data.loc[eligible].copy()

## 3. Start or load a company project

In [ ]:
project = None
for name in ["current_news", "daily_features", "training_candidates", "sentiment_evaluation"]:
    globals().pop(name, None)
project = choose_project()
sync_aliases(project)

## 4. Refresh or reuse market data

In [ ]:
if not project:
    raise RuntimeError("Load a project first.")
refresh = not project.get("market_fetch_attempted") or ask_yes_no("Refresh market data? [y/n]: ")
if refresh:
    updated = fetch_market(project["config"])
    project["market_fetch_attempted"] = True
    if not updated.empty:

        market = pd.concat([project["market"], updated]).sort_index()
        market = market.loc[~market.index.duplicated(keep="last")].copy()
        for c in ["Company", "Target Ticker", "Run ID"]:
            market = market.drop(columns=c, errors="ignore")
        market["MA5"] = market["Close"].rolling(5).mean()
        market["MA20"] = market["Close"].rolling(20).mean()
        market["VolumeMA20"] = market["Volume"].rolling(20).mean()
        market["DailyReturn"] = market["Close"].pct_change(fill_method=None)
        market["IntradayReturn"] = market["Close"] / market["Open"] - 1
        project["market"] = market
    else:
        print("Existing market data retained.")
    sync_aliases(project)
    save_project(project)
plot_market(project["market"], project["config"])

## 5. Retrieve news and add your own sources
Links are stored; webpage content is not automatically downloaded. Paste a title or excerpt to analyze it.

In [ ]:
if not project:
    raise RuntimeError("Load a project first.")
refresh = not project.get("news_fetch_attempted") or ask_yes_no("Refresh news? [y/n]: ")
if refresh:
    updated = fetch_news(project["config"])
    project["news_fetch_attempted"] = True
    if not updated.empty:
        project["news_raw"] = merge_news(project["news_raw"], updated)
        project["review_complete"] = False
    else:
        print("Existing news and decisions retained.")
    save_project(project)
additions = collect_additional_sources()
if not additions.empty:
    project["manual_sources"] = merge_news(project["manual_sources"], additions)
    project["review_complete"] = False
    save_project(project)
sync_aliases(project)

## 6. Review company relevance
Unchanged accepted and rejected decisions are reused. New or changed references require review. Q pauses; each decision is saved. All candidates are reviewed, including indirect company events.

In [ ]:
review_project_sources(project)

## 7. Score new or changed articles
Existing scores and human reviews remain intact. Long texts are truncated to 512 model tokens.

In [ ]:
score_reviewed_news(project)

## 8. Optional human sentiment review
Review financial tone before seeing model labels. This does not train FinBERT or alter model scores.

In [ ]:
if ask_yes_no("Review sentiment manually? [y/n]: "):
    review_news_sentiment(project)

## 9. Current news summary
The figure uses model scores from all eligible articles; the table previews five.

In [ ]:
current_news = show_current_news_summary(project, freshness_hours=72)

## 10. Daily features and training readiness
Acquired news is never moved backward into historical dates. No-news collection windows do not prove absence of market news.

In [ ]:
daily_features = build_daily_features(project, freshness_hours=72)
training_candidates = check_training_readiness(daily_features)

## 11. Optional label evaluation and final save
Small, selectively reviewed samples do not estimate general model accuracy. Derived tables are rebuilt, while sources, scores and decisions are saved.

In [ ]:
if ask_yes_no("Show model versus human label evaluation? [y/n]: "):
    sentiment_evaluation = evaluate_sentiment_reviews(project)
save_project(project)
print("Next time, load this run:", project["run_id"])